# 02 - Fine-tune Llama 3.1 8B with QLoRA

Fine-tune `meta-llama/Llama-3.1-8B-Instruct` on Vietnamese CSKH data.

### Advanced techniques applied:
- **QLoRA** (4-bit NF4 + double quantization)
- **Gradient checkpointing** (saves ~40% VRAM)
- **NEFTune** (noisy embeddings, +10-15% chat quality — arxiv:2310.05914)
- **Packing** (fills max_seq_length for GPU efficiency)
- **Cosine LR schedule** with warmup
- **paged_adamw_8bit** optimizer (VRAM-efficient)
- **Early stopping** (patience=3, prevents overfitting)
- **Google Drive checkpointing** (survive Colab disconnects)
- **Auto-resume** from last checkpoint

### Requirements
- Colab Pro with **A100 GPU** (~15-21 compute units)
- HuggingFace account with Llama 3.1 access

### Output
- GGUF Q3_K_M (~3.9GB) for Ollama on RTX 3060 6GB

In [ ]:
# Cell 1: Install dependencies
# ⚠️ SAU KHI CHẠY XONG CELL NÀY → Runtime → Restart session → rồi chạy từ Cell 2
!pip install -q \
    "huggingface_hub>=0.25" \
    "transformers>=4.44,<5" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "peft>=0.13" \
    "bitsandbytes>=0.43" \
    "trl>=0.11"

print('✅ Done! Bây giờ hãy: Runtime → Restart session → rồi chạy từ Cell 2')

In [ ]:
# Cell 2: Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/vani-copilot'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
# Cell 3: Check GPU and VRAM
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → GPU (A100)')

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu}')
print(f'VRAM: {vram:.1f} GB')

if vram < 15:
    print(f'⚠️ Only {vram:.0f}GB VRAM — A100 (40GB) strongly recommended')
    print('  T4 (16GB) might work but will be very tight')
    print('  Go to: Runtime → Change runtime type → A100')
elif 'A100' in gpu or 'H100' in gpu:
    print('✅ Perfect GPU for QLoRA training')
else:
    print(f'✅ {vram:.0f}GB should be sufficient')

In [ ]:
# Cell 4: Login to HuggingFace (TÙY CHỌN)
# Đã dùng unsloth mirror (ungated) → KHÔNG CẦN LOGIN, bỏ qua cell này.
# Chỉ cần nếu dùng model gốc meta-llama/ (gated).

# from huggingface_hub import login
# login()

In [ ]:
# Cell 5: Configuration - ALL hyperparameters in one place
import os

# Ungated mirror — không cần xin quyền từ Meta
MODEL_ID = 'unsloth/Meta-Llama-3.1-8B-Instruct'

# Paths
LOCAL_OUTPUT = './vani-qlora-checkpoints'
DRIVE_CHECKPOINT = os.path.join(DRIVE_DIR, 'qlora-checkpoints')
MERGED_DIR = './vani-copilot-merged'

# QLoRA config
LORA_R = 64              # rank - higher = more capacity, more VRAM
LORA_ALPHA = 128         # scaling factor (alpha/r = 2 is a good ratio)
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention
    'gate_proj', 'up_proj', 'down_proj',       # MLP
]

# Training config
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4   # effective batch = 4 * 4 = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

# Advanced
NEFTUNE_NOISE_ALPHA = 5.0   # NEFTune: adds noise to embeddings during training
USE_PACKING = True          # pack short sequences together

# Checkpointing
SAVE_STEPS = 100
EVAL_STEPS = 100
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 3       # keep last 3 checkpoints to save disk

print('Config set.')
print(f'Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}')

In [ ]:
# Cell 6: Load training data from Drive
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

# Verify files exist
for f in ['train.jsonl', 'val.jsonl']:
    p = os.path.join(DRIVE_DIR, f)
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'❌ {f} not found in {DRIVE_DIR}\n'
            '👉 Upload colab-upload/ files to MyDrive/vani-copilot/'
        )
    print(f'✅ {f}: {os.path.getsize(p):,} bytes')

train_raw = load_jsonl(os.path.join(DRIVE_DIR, 'train.jsonl'))
val_raw = load_jsonl(os.path.join(DRIVE_DIR, 'val.jsonl'))

# Validate format
sample = train_raw[0]
assert 'messages' in sample, f'Bad format: expected "messages" key, got {list(sample.keys())}'
assert len(sample['messages']) >= 2, 'Each sample needs at least 2 messages'

print(f'\nTrain: {len(train_raw)} samples')
print(f'Val: {len(val_raw)} samples')
print(f'Roles in sample: {[m["role"] for m in sample["messages"]]}')
print(f'\nFirst message preview:')
print(json.dumps(sample, ensure_ascii=False, indent=2)[:500])

In [ ]:
# Cell 7: Load model with 4-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,  # nested quantization saves extra VRAM
)

# Try Flash Attention 2 for speed, fallback to eager
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2',
    )
    print('Using Flash Attention 2')
except Exception:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
    )
    print('Using default attention (Flash Attention not available)')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model: {MODEL_ID}')
print(f'Parameters: {model.num_parameters():,}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Cell 8: Apply LoRA with gradient checkpointing
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,  # saves ~40% VRAM
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'VRAM after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Cell 9: Format data with chat template
import random as _rnd

def format_chat(sample):
    text = tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_dataset = Dataset.from_list(train_raw).map(format_chat, num_proc=1)
val_dataset = Dataset.from_list(val_raw).map(format_chat, num_proc=1)

# Token length analysis on a sample (saves time + RAM)
_sample_idx = _rnd.sample(range(len(train_dataset)), min(500, len(train_dataset)))
sample_lengths = [len(tokenizer.encode(train_dataset[i]['text'])) for i in _sample_idx]
over_limit = sum(1 for l in sample_lengths if l > MAX_SEQ_LENGTH)

print(f'Train: {len(train_dataset)} samples')
print(f'Val: {len(val_dataset)} samples')
print(f'\nToken lengths (sampled {len(sample_lengths)}):')
print(f'  min={min(sample_lengths)}, max={max(sample_lengths)}, avg={sum(sample_lengths)/len(sample_lengths):.0f}')
print(f'  > {MAX_SEQ_LENGTH} tokens: {over_limit} ({over_limit/len(sample_lengths)*100:.1f}%) — will be truncated/packed')
print(f'\nSample formatted text:')
print(train_dataset[0]['text'][:500])

In [ ]:
# Cell 10: Check for existing checkpoint (auto-resume)
import glob

resume_checkpoint = None

# Check Drive first, then local
for check_dir in [DRIVE_CHECKPOINT, LOCAL_OUTPUT]:
    if os.path.exists(check_dir):
        checkpoints = sorted(glob.glob(os.path.join(check_dir, 'checkpoint-*')))
        if checkpoints:
            resume_checkpoint = checkpoints[-1]
            print(f'FOUND CHECKPOINT: {resume_checkpoint}')
            print(f'Will RESUME training from this checkpoint.')
            break

if not resume_checkpoint:
    print('No checkpoint found. Starting fresh training.')

In [ ]:
# Cell 11: Training with all optimizations
import shutil
import gc
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, EarlyStoppingCallback

gc.collect()
torch.cuda.empty_cache()

# Custom callback: sync checkpoints to Drive after each save
class DriveCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        """Copy latest checkpoint to Drive after each save."""
        src = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        if not os.path.exists(src):
            return
        dst = os.path.join(DRIVE_CHECKPOINT, f'checkpoint-{state.global_step}')
        if not os.path.exists(dst):
            try:
                shutil.copytree(src, dst)
                print(f'\n💾 Checkpoint synced to Drive: step {state.global_step}')
            except Exception as e:
                print(f'\n⚠️ Drive sync failed: {e}')

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Print VRAM usage periodically."""
        if state.global_step % 50 == 0 and state.global_step > 0:
            vram = torch.cuda.memory_allocated() / 1e9
            print(f'  [step {state.global_step}] VRAM: {vram:.1f}GB')

training_args = SFTConfig(
    output_dir=LOCAL_OUTPUT,

    # Core training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    # Learning rate
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',

    # Sequence
    max_seq_length=MAX_SEQ_LENGTH,
    packing=USE_PACKING,
    dataset_text_field='text',

    # NEFTune: adds uniform noise to embedding vectors during training
    # Proven to improve chat model quality by 10-15% (arxiv:2310.05914)
    neftune_noise_alpha=NEFTUNE_NOISE_ALPHA,

    # Precision & optimization
    bf16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},

    # Evaluation
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Checkpointing
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,

    # Logging
    logging_steps=LOGGING_STEPS,
    report_to='none',

    # Misc — num_workers=0 prevents RAM spike on Colab
    seed=42,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        DriveCheckpointCallback(),
    ],
)

print(f'Training config:')
print(f'  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}')
print(f'  NEFTune alpha: {NEFTUNE_NOISE_ALPHA}')
print(f'  Packing: {USE_PACKING}')
print(f'  Gradient checkpointing: True')
print(f'  Early stopping: patience=3')
print(f'  Checkpoint every {SAVE_STEPS} steps → auto-sync to Drive')
print(f'  Resume from: {resume_checkpoint or "scratch"}')
print(f'  VRAM before training: {torch.cuda.memory_allocated() / 1e9:.1f}GB')

In [ ]:
# Cell 12: START TRAINING (auto-resumes if checkpoint found)
print('='*60)
print('STARTING TRAINING')
print(f'Resume: {resume_checkpoint or "fresh start"}')
print('='*60)

try:
    result = trainer.train(resume_from_checkpoint=resume_checkpoint)
    print('\n' + '='*60)
    print('✅ TRAINING COMPLETE!')
    print(f'  Total steps: {result.global_step}')
    print(f'  Train loss: {result.training_loss:.4f}')
    if trainer.state.best_metric is not None:
        print(f'  Best eval loss: {trainer.state.best_metric:.4f}')
    print('='*60)
except KeyboardInterrupt:
    print('\n⚠️ Training interrupted! Checkpoints are saved on Drive.')
    print('  Re-run this cell to resume from last checkpoint.')
except Exception as e:
    print(f'\n❌ Training error: {e}')
    print('  Checkpoints should be on Drive. Re-run cell 10 + 12 to resume.')
    raise

In [ ]:
# Cell 13: Save LoRA adapter to Drive (backup!)
import shutil

adapter_dir = os.path.join(DRIVE_DIR, 'qlora-adapter-final')
os.makedirs(adapter_dir, exist_ok=True)

trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Verify adapter was saved
adapter_files = os.listdir(adapter_dir)
print(f'✅ Adapter saved to: {adapter_dir}')
print(f'   Files: {adapter_files}')

# Copy any remaining local checkpoints to Drive
if os.path.exists(LOCAL_OUTPUT):
    os.makedirs(DRIVE_CHECKPOINT, exist_ok=True)
    for ckpt in sorted(glob.glob(os.path.join(LOCAL_OUTPUT, 'checkpoint-*'))):
        dest = os.path.join(DRIVE_CHECKPOINT, os.path.basename(ckpt))
        if not os.path.exists(dest):
            shutil.copytree(ckpt, dest)
            print(f'   Synced: {os.path.basename(ckpt)}')

print('\n✅ Everything backed up to Google Drive!')

In [ ]:
# Cell 14: Merge LoRA into base model
import gc
from peft import PeftModel

# Free VRAM from training
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM freed. Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f}GB')

# Reload base model in bf16 for merging
print('Loading base model for merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)

print(f'Loading adapter from {adapter_dir}...')
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Verify
merge_size = sum(os.path.getsize(os.path.join(MERGED_DIR, f)) for f in os.listdir(MERGED_DIR)) / 1e9
print(f'✅ Merged model saved to {MERGED_DIR} ({merge_size:.1f}GB)')

In [ ]:
# Cell 15: Test the fine-tuned model before GGUF conversion
from transformers import pipeline

pipe = pipeline(
    'text-generation',
    model=merged_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)

test_cases = [
    'Shop ơi em cao 1m58 nặng 49kg mặc váy midi xếp ly size gì ạ?',
    'Cho hỏi ship nội thành bao lâu vậy shop?',
    'Hàng bị lỗi thì đổi trả thế nào ạ?',
    'Có mã giảm giá gì không shop?',
]

system = 'Bạn là nhân viên CSKH của VANI Store - shop thời trang nữ online. Xưng em, gọi khách là chị/anh. Thân thiện, lịch sự.'

for q in test_cases:
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': q},
    ]
    result = pipe(messages)
    answer = result[0]['generated_text'][-1]['content']
    print(f'\nQ: {q}')
    print(f'A: {answer}')
    print('-' * 60)

In [ ]:
# Cell 16: Build GGUF converter
import os

if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
else:
    print('llama.cpp already cloned')

# Build quantizer
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release 2>/dev/null && cmake --build build --target llama-quantize -j$(nproc) 2>/dev/null

# Fallback to make if cmake fails
import os
quantize_path = 'llama.cpp/build/bin/llama-quantize'
if not os.path.exists(quantize_path):
    !cd llama.cpp && make -j$(nproc) llama-quantize 2>/dev/null
    quantize_path = 'llama.cpp/llama-quantize'

# Install Python converter deps
!pip install -q gguf sentencepiece protobuf

print(f'✅ Quantizer ready at: {quantize_path}')
print(f'   Exists: {os.path.exists(quantize_path)}')

In [ ]:
# Cell 17: Convert HF → GGUF → Quantize
import gc

# Free VRAM for conversion
try:
    del merged_model, base_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Step 1: HF → GGUF F16
print('Step 1/2: Converting HF model to GGUF F16...')
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile vani-copilot-f16.gguf --outtype f16

if not os.path.exists('vani-copilot-f16.gguf'):
    raise RuntimeError('F16 GGUF conversion failed! Check output above.')

f16_size = os.path.getsize('vani-copilot-f16.gguf') / 1e9
print(f'✅ F16 GGUF: {f16_size:.1f}GB')

# Step 2: Quantize to Q3_K_M (optimized for RTX 3060 6GB)
print('\nStep 2/2: Quantizing to Q3_K_M...')
!{quantize_path} vani-copilot-f16.gguf vani-copilot-q3_k_m.gguf Q3_K_M

if not os.path.exists('vani-copilot-q3_k_m.gguf'):
    raise RuntimeError('Quantization failed! Check output above.')

size_mb = os.path.getsize('vani-copilot-q3_k_m.gguf') / 1e6
print(f'\n✅ GGUF Q3_K_M: {size_mb:.0f} MB')
print(f'   Estimated VRAM usage: ~{size_mb * 1.15:.0f} MB')
print(f'   RTX 3060 6GB: {"✅ FITS" if size_mb < 5000 else "⚠️ TIGHT"}')

In [ ]:
# Cell 18: Save GGUF to Drive (MOST IMPORTANT FILE!)
import shutil

gguf_dest = os.path.join(DRIVE_DIR, 'vani-copilot-q3_k_m.gguf')
print(f'Copying GGUF to Drive (this may take a minute)...')
shutil.copy2('vani-copilot-q3_k_m.gguf', gguf_dest)

# Verify
src_size = os.path.getsize('vani-copilot-q3_k_m.gguf')
dst_size = os.path.getsize(gguf_dest)
assert src_size == dst_size, f'Copy verification failed! {src_size} != {dst_size}'

print(f'✅ GGUF saved to Drive: {gguf_dest}')
print(f'   Size: {dst_size / 1e6:.0f} MB (verified)')
print()
print('='*60)
print('NEXT STEPS:')
print('  1. Download vani-copilot-q3_k_m.gguf from Drive')
print('  2. Place it in backend/ folder')
print('  3. Create Modelfile + run: ollama create vani-copilot -f Modelfile')
print('  4. Set LLM_PROVIDER=ollama in .env')
print('='*60)

In [ ]:
# Cell 19: (Optional) Push to HuggingFace Hub
import os
from huggingface_hub import HfApi

REPO_ID = 'jot2003/vani-copilot-llama3.1-8b-qlora'

# File GGUF có thể nằm ở local hoặc đã copy sang Drive
gguf_path = 'vani-copilot-q3_k_m.gguf'
if not os.path.exists(gguf_path):
    gguf_path = os.path.join(DRIVE_DIR, 'vani-copilot-q3_k_m.gguf')
if not os.path.exists(gguf_path):
    raise FileNotFoundError(f'GGUF file not found! Check cell 17-18 output.')

print(f'Uploading: {gguf_path} ({os.path.getsize(gguf_path)/1e6:.0f} MB)')

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj=gguf_path,
    path_in_repo='vani-copilot-q3_k_m.gguf',
    repo_id=REPO_ID,
)
print(f'✅ Uploaded to https://huggingface.co/{REPO_ID}')